# Wetland Loss to Agriculture in a Prairie Pothole Watershed, 2000–2020

**Question:** How much wetland in the Pipestem watershed (HUC8, North Dakota) was converted to cropland between 2000 and 2020?

**Approach in one sentence:** build cloud-free spring Landsat composites for 2000, 2010 and 2020, train a separate
random forest for each snapshot using that period's Cropland Data Layer and the National Wetlands Inventory as labels,
classify each snapshot, and track pixels that went from wetland to cropland.

**Why 2000–2020 and not 1990?** A first version trained one model on 2020 and applied it to 1990, 2000 and 2010.
Sanity checks showed that model did not carry over to older imagery: it mapped implausibly large wetland areas in the
earlier years, including field-shaped "wetlands" during the 1989–1991 drought. Testing variants (with and without a
Landsat 8 → Landsat 7 adjustment, and with indices only) did not fix it (`scripts/diagnose_transfer.py`). The Cropland
Data Layer covers this watershed every year from 1997, so restricting the study to 2000–2020 lets every snapshot have
**its own training labels and its own model**.

Every number reported in this project comes from a cell in this notebook.

## Step 0 — Connect to Earth Engine

Earth Engine runs the heavy processing on Google's servers; this notebook just sends it instructions.
To use it you need a (free, noncommercial) Google Cloud project registered for Earth Engine.
`ee.Initialize(project=...)` tells Earth Engine which project to bill the (free) usage to.

In [1]:
import ee
import geemap

# Your Google Cloud project registered for noncommercial Earth Engine use
PROJECT_ID = "agriculture-wetland-loss"

# Authenticate() only opens a browser the first time; after that it reuses saved credentials.
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized:", ee.String("ok").getInfo())

Earth Engine initialized: ok


## Step 1a — Load the watershed boundary

The USGS Watershed Boundary Dataset (WBD) divides the U.S. into nested hydrologic units.
A **HUC8** is a sub-basin, typically a few thousand km². We look up Pipestem by name and print its
code and area so we can confirm we have the right polygon before doing anything else.

In [2]:
huc8 = ee.FeatureCollection("USGS/WBD/2017/HUC08")
watershed = huc8.filter(ee.Filter.eq("name", "Pipestem"))

# Check what we actually got (there should be exactly one feature, in ND)
info = watershed.first().toDictionary().getInfo()
print("Features matched:", watershed.size().getInfo())
print({k: info[k] for k in sorted(info)})

aoi = watershed.geometry()  # the area of interest used everywhere below
print("Area computed from geometry (km²):", round(aoi.area(maxError=1).divide(1e6).getInfo(), 1))

Features matched: 1
{'areaacres': '685100.43', 'areasqkm': '2772.51', 'gnis_id': '', 'huc8': '10160002', 'loaddate': '20120611075457', 'metasource': '', 'name': 'Pipestem', 'shape_area': '0.329154073822313', 'shape_leng': '4.36477122786687', 'sourcedata': '', 'sourcefeat': '', 'sourceorig': '', 'states': 'ND', 'tnmid': '{C1EC565A-3D79-4C86-9D4E-B1DE84BABDB3}'}


Area computed from geometry (km²): 2764.9


## Step 1b — Cloud masking, scaling, and a common band naming

**Landsat Collection 2 Level-2** gives *surface reflectance*: the fraction of sunlight reflected by the ground,
with atmospheric effects removed. A few housekeeping steps are needed for every image:

1. **Cloud mask.** Each scene has a `QA_PIXEL` band where individual bits flag fill, cloud, dilated cloud,
   cloud shadow and snow. We drop any pixel with one of those flags, plus saturated pixels (`QA_RADSAT`).
2. **Scale factors.** Reflectance is stored as integers; `value × 0.0000275 − 0.2` converts to 0–1 reflectance.
3. **Common band names.** Landsat 5/7 and Landsat 8 number their bands differently
   (e.g. red is `SR_B3` on Landsat 5/7 but `SR_B4` on Landsat 8), so we rename to `blue, green, red, nir, swir1, swir2`.

Note: Landsat 8 sees slightly different wavelengths than Landsat 5/7. We use its surface reflectance as-is
(no cross-sensor adjustment), which keeps the method simple; the possible effect is noted under limitations in the README.

In [3]:
BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
TM_ETM_BANDS = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"]  # Landsat 5 and 7
OLI_BANDS = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]     # Landsat 8


def mask_clouds(img):
    """Keep only clear pixels using the QA_PIXEL bit flags."""
    qa = img.select("QA_PIXEL")
    # bit 0 fill, bit 1 dilated cloud, bit 3 cloud, bit 4 cloud shadow, bit 5 snow
    bad = (1 << 0) | (1 << 1) | (1 << 3) | (1 << 4) | (1 << 5)
    clear = qa.bitwiseAnd(bad).eq(0)
    not_saturated = img.select("QA_RADSAT").eq(0)
    return img.updateMask(clear).updateMask(not_saturated)


def scale(img, band_names):
    """Convert stored integers to surface reflectance and rename to common names."""
    return img.select(band_names, BANDS).multiply(0.0000275).add(-0.2)


def prep_tm_etm(img):
    return scale(mask_clouds(img), TM_ETM_BANDS).copyProperties(img, ["system:time_start"])


def prep_oli(img):
    return scale(mask_clouds(img), OLI_BANDS).copyProperties(img, ["system:time_start"])

## Step 1c — Build the three spring composites

For each snapshot year we take **every clear Landsat pixel from April–June over a 3-year window** and keep the
**median** value per pixel. The median ignores leftover clouds or odd values, and the 3-year window
smooths out a single unusually wet or dry spring.

| Snapshot | Window | Sensors |
|---|---|---|
| 2000 | 1999–2001 | Landsat 5 + 7 |
| 2010 | 2009–2011 | Landsat 5 + 7 (Landsat 7 has striping gaps after 2003; other scenes fill them) |
| 2020 | 2019–2021 | Landsat 8 |

We print how many scenes went in and how many clear observations each pixel had, so we know the composites are well supported.

In [4]:
COLLECTIONS = {
    "L5": ("LANDSAT/LT05/C02/T1_L2", prep_tm_etm),
    "L7": ("LANDSAT/LE07/C02/T1_L2", prep_tm_etm),
    "L8": ("LANDSAT/LC08/C02/T1_L2", prep_oli),
}

SNAPSHOTS = {
    2000: (1999, 2001, ["L5", "L7"]),
    2010: (2009, 2011, ["L5", "L7"]),
    2020: (2019, 2021, ["L8"]),
}


def spring_collection(first_year, last_year, sensors):
    """All cloud-masked April-June scenes over the AOI for the window, merged across sensors."""
    merged = ee.ImageCollection([])
    for s in sensors:
        collection_id, prep = COLLECTIONS[s]
        col = (
            ee.ImageCollection(collection_id)
            .filterBounds(aoi)
            .filterDate(f"{first_year}-01-01", f"{last_year + 1}-01-01")
            .filter(ee.Filter.calendarRange(4, 6, "month"))
            .map(prep)
        )
        merged = merged.merge(col)
    return merged


composites = {}
for year, (y0, y1, sensors) in SNAPSHOTS.items():
    col = spring_collection(y0, y1, sensors)
    composites[year] = col.median().clip(aoi)

    # Clear observations per pixel = how many scenes had a usable value there
    n_clear = col.select("red").count().clip(aoi)
    stats = n_clear.reduceRegion(
        reducer=ee.Reducer.min().combine(ee.Reducer.mean(), sharedInputs=True),
        geometry=aoi, scale=30, maxPixels=1e10,
    ).getInfo()
    print(f"{year} ({y0}-{y1}, {'+'.join(sensors)}): {col.size().getInfo()} scenes | "
          f"clear obs per pixel: min {stats['red_min']}, mean {stats['red_mean']:.1f}")

2000 (1999-2001, L5+L7): 43 scenes | clear obs per pixel: min 1, mean 14.5


2010 (2009-2011, L5+L7): 57 scenes | clear obs per pixel: min 2, mean 23.6


2020 (2019-2021, L8): 45 scenes | clear obs per pixel: min 4, mean 13.2


## Step 1d — Look at the composites

Two views per year:
- **True color** (red, green, blue): what your eye would see.
- **False color** (SWIR1, NIR, red): vegetation shows green, open water and wet soils show dark blue/black, and bare
  fields look pink/brown. This is the easiest way to spot prairie potholes.

Toggle layers in the top-right control to compare years.

In [5]:
m = geemap.Map(basemap="Esri.WorldImagery")
m.centerObject(aoi, 10)

true_color = {"bands": ["red", "green", "blue"], "min": 0, "max": 0.2}
false_color = {"bands": ["swir1", "nir", "red"], "min": 0, "max": 0.35}

for year, img in composites.items():
    m.addLayer(img, true_color, f"{year} true color", shown=False)
    m.addLayer(img, false_color, f"{year} false color", shown=(year in (2000, 2020)))

m.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m

## Step 2a — Add spectral indices

Raw bands are useful, but a few **indices** (simple band math) make the three classes much easier to separate:

| Index | Formula | What it highlights |
|---|---|---|
| **NDVI** (Normalized Difference Vegetation Index) | (NIR − Red) / (NIR + Red) | Green, growing vegetation is high; water is negative; bare spring fields are low. |
| **MNDWI** (Modified Normalized Difference Water Index, Xu 2006) | (Green − SWIR1) / (Green + SWIR1) | Open water is positive; land is mostly negative. Uses SWIR1 because water absorbs it strongly. |
| **Tasseled Cap Wetness** (Crist 1985) | weighted sum of all six bands | Soil and plant moisture. Higher = wetter. Helps with shallow, vegetated wetlands that aren't open water. |

Spring timing helps here: in April–June many crop fields are still bare or newly planted (low NDVI), while
grassland is green and potholes hold snowmelt water.

The wetness coefficients were derived for Landsat 5 TM reflectance. We apply them to all sensors unchanged,
consistent with our decision not to adjust Landsat 8 (noted in the README limitations).

In [6]:
# Crist (1985) Tasseled Cap wetness coefficients for TM reflectance,
# in the order blue, green, red, nir, swir1, swir2
TC_WETNESS = [0.0315, 0.2021, 0.3102, 0.1594, -0.6806, -0.6109]


def add_indices(img):
    """Return the composite with NDVI, MNDWI and Tasseled Cap wetness bands added."""
    ndvi = img.normalizedDifference(["nir", "red"]).rename("ndvi")
    mndwi = img.normalizedDifference(["green", "swir1"]).rename("mndwi")
    # Multiply each band by its coefficient, then add them up across bands
    tcw = (
        img.select(BANDS)
        .multiply(ee.Image.constant(TC_WETNESS))
        .reduce(ee.Reducer.sum())
        .rename("tcw")
    )
    return img.addBands([ndvi, mndwi, tcw])


composites = {year: add_indices(img) for year, img in composites.items()}
print("Bands now in each composite:", composites[2020].bandNames().getInfo())

Bands now in each composite: ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'mndwi', 'tcw']


## Step 2b — Sanity check: index summaries per snapshot

For each snapshot we print the 10th percentile, median and 90th percentile of each index across the watershed,
plus the share of pixels with **MNDWI > 0**. That last number is only a rough "looks like open water" indicator
(no classification yet), but it shows how wet each window was before we classify anything.

In [7]:
import pandas as pd

rows = []
for year, img in composites.items():
    pct = img.select(["ndvi", "mndwi", "tcw"]).reduceRegion(
        reducer=ee.Reducer.percentile([10, 50, 90]),
        geometry=aoi, scale=30, maxPixels=1e10,
    )
    # Mean of a 0/1 image = fraction of pixels where the condition is true
    water_like = img.select("mndwi").gt(0).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e10,
    )
    row = {"year": year, **pct.getInfo()}
    row["pct_pixels_mndwi_gt_0"] = water_like.get("mndwi").getInfo() * 100
    rows.append(row)

index_summary = pd.DataFrame(rows).set_index("year")
# Order columns as index_p10, index_p50, index_p90
cols = [f"{b}_p{p}" for b in ["ndvi", "mndwi", "tcw"] for p in (10, 50, 90)] + ["pct_pixels_mndwi_gt_0"]
index_summary = index_summary[cols].round(3)
index_summary

,ndvi_p10,ndvi_p50,ndvi_p90,mndwi_p10,mndwi_p50,mndwi_p90,tcw_p10,tcw_p50,tcw_p90,pct_pixels_mndwi_gt_0
year,,,,,,,,,,
2000,0.223,0.363,0.566,-0.551,-0.488,-0.356,-0.287,-0.201,-0.092,4.867
2010,0.184,0.293,0.535,-0.543,-0.504,-0.363,-0.319,-0.214,-0.089,5.059
2020,0.231,0.332,0.551,-0.566,-0.527,-0.434,-0.330,-0.252,-0.123,5.187


## Step 2c — Look at the indices on a map

Blue = wet / water, brown = dry, green = vegetated. Compare 2000 and 2020 for each index with the layer control.

In [8]:
m2 = geemap.Map(basemap="Esri.WorldImagery")
m2.centerObject(aoi, 10)

ndvi_vis = {"min": -0.2, "max": 0.6, "palette": ["8c510a", "f6e8c3", "1a9850"]}
mndwi_vis = {"min": -0.6, "max": 0.4, "palette": ["8c510a", "f5f5f5", "2166ac"]}
tcw_vis = {"min": -0.35, "max": 0.0, "palette": ["8c510a", "f5f5f5", "2166ac"]}

for year in (2000, 2020):
    img = composites[year]
    m2.addLayer(img.select("ndvi"), ndvi_vis, f"{year} NDVI", shown=False)
    m2.addLayer(img.select("tcw"), tcw_vis, f"{year} TC wetness", shown=False)
    m2.addLayer(img.select("mndwi"), mndwi_vis, f"{year} MNDWI", shown=(year == 2020))

m2.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m2

## Step 3a — Reference data: NWI wetlands and the Cropland Data Layer

To train a classifier we need **labelled examples**: pixels where we already know the answer. Instead of
hand-digitizing, we borrow labels from two authoritative public datasets:

- **National Wetlands Inventory (NWI)**, USFWS. Wetland polygons for Pipestem were mapped from 2017–2019
  imagery. We downloaded the USFWS HUC8 file, clipped it to the watershed with GeoPandas
  (`scripts/prepare_nwi.py`), and uploaded it as an Earth Engine asset.
  Polygons coded `Pf` (*palustrine farmed*) are wetlands that are being cropped; we flag them with `farmed = 1`.
- **Cropland Data Layer (CDL)**, USDA NASS. A yearly 30 m crop-type map, available for this watershed every year
  since 1997. Its `cropland` band gives a crop-type code per pixel (e.g. 5 = soybeans, 61 = fallow/idle, 176 = grass/pasture).

**Defining "cultivated" the same way for every year.** Recent CDL years also have a ready-made `cultivated` band,
but older years (before 2008) don't. So we define cultivated ourselves from the crop codes (1–61, 66–77 and
204–254 are crops, including fallow/idle) and use that rule for all years. Below we check that our rule matches
the official `cultivated` band in 2020.

We also print the 2020 CDL class breakdown to see what the landscape looks like.

In [9]:
NWI_ASSET = "projects/agriculture-wetland-loss/assets/nwi_pipestem"
nwi = ee.FeatureCollection(NWI_ASSET)
print("NWI polygons:", nwi.size().getInfo(), "| farmed (Pf):", nwi.filter(ee.Filter.eq("farmed", 1)).size().getInfo())


def cdl_cropland(year):
    """The CDL crop-type band for one year."""
    return ee.Image(f"USDA/NASS/CDL/{year}").select("cropland")


def is_cultivated(crop):
    """1 where the CDL code is a crop (including fallow/idle, 61), 0 otherwise."""
    return (crop.gte(1).And(crop.lte(61))
            .Or(crop.gte(66).And(crop.lte(77)))
            .Or(crop.gte(204).And(crop.lte(254))))


# CDL water and wetland codes, old legend (83 water, 87 wetlands) and new legend (111, 190, 195)
WATER_WETLAND_CODES = [83, 87, 111, 190, 195]

cdl_proj = cdl_cropland(2020).projection()  # CDL's native 30 m Albers (EPSG:5070) grid; we sample on this grid

# Check: does our code-based rule match the official 2020 "cultivated" band (2 = cultivated)?
official = ee.Image("USDA/NASS/CDL/2020").select("cultivated").eq(2)
agreement = is_cultivated(cdl_cropland(2020)).eq(official).reduceRegion(
    ee.Reducer.mean(), aoi, crs=cdl_proj, scale=30, maxPixels=1e10).getInfo()
print(f"Our 'cultivated' rule agrees with the official 2020 cultivated band on {100 * list(agreement.values())[0]:.2f}% of pixels")

# 2020 CDL class breakdown inside the watershed
hist = cdl_cropland(2020).reduceRegion(
    ee.Reducer.frequencyHistogram(), aoi, scale=30, maxPixels=1e10
).get("cropland").getInfo()
# The class codes and names are stored as comma-separated strings in the image metadata
codes = ee.Image("USDA/NASS/CDL/2020").get("cropland_class_values").getInfo().split(",")
names = ee.Image("USDA/NASS/CDL/2020").get("cropland_class_names").getInfo().split(",")
assert len(codes) == len(names), "CDL code/name lists don't line up"
code_to_name = dict(zip(codes, names))

cdl_breakdown = (
    pd.DataFrame({"code": list(hist), "pixels": list(hist.values())})
    .assign(name=lambda d: d.code.map(code_to_name),
            percent=lambda d: (100 * d.pixels / d.pixels.sum()).round(1))
    .sort_values("pixels", ascending=False)
    .head(12)
    .set_index("code")
)
cdl_breakdown

NWI polygons: 44854 | farmed (Pf): 9191


Our 'cultivated' rule agrees with the official 2020 cultivated band on 91.80% of pixels


,pixels,name,percent
code,,,
176,746567.674510,Grass/Pasture,24.2
5,682063.427451,Soybeans,22.1
61,605600.266667,Fallow/Idle Cropland,19.7
195,231480.737255,Herbaceous Wetlands,7.5
111,221370.552941,Open Water,7.2
23,177753.388235,Spring Wheat,5.8
1,117718.709804,Corn,3.8
121,76274.082353,Developed/Open Space,2.5
37,60845.149020,Other Hay/Non Alfalfa,2.0


## Step 3b — Build "confident" label masks for each snapshot

Each snapshot gets its own labels, built from the CDL for the **three years of its window** (e.g. 1999–2001 for 2000)
plus NWI. Good training labels matter more than lots of them, so each class has strict rules. A pixel only becomes
a candidate if **all** of its rules hold:

| Class | Rule (CDL years = the snapshot's 3-year window) |
|---|---|
| **0 — Wetland/water** | Inside a non-farmed NWI polygon of the right type (below), **and** not cropped in any of the CDL years (fallow/idle allowed, see below) |
| **1 — Cropland** | Cultivated in all three CDL years, **not** fallow/idle (61) in any of them, and outside all NWI polygons |
| **2 — Grassland/other** | Not cultivated in any CDL year, not CDL water/wetland in any of them, and outside all NWI polygons |

**Two kinds of wetland.** Prairie potholes range from open water to marsh full of cattails and sedges.
If we only sampled "wetland" at random, most points would land in big lakes (the easiest pixels to keep),
and the model would learn *wetland = open water*. So the wetland class is built from two **strata**
(sub-groups), sampled separately and then merged into one class:
- **open water**: NWI `Lake` and `Freshwater Pond`
- **emergent marsh**: NWI `Freshwater Emergent Wetland`

Extra choices, all to avoid mislabelled pixels:
- **Three CDL years per snapshot** filters out one-year CDL errors and crop rotations into grass.
- **Smoothing the CDL first.** Older CDL years are "speckled": scattered single pixels with a different code
  inside a uniform field or marsh. Before applying the rules we pass a 3×3 **majority filter** over each CDL year
  (each pixel takes the most common code around it), which removes the speckle but keeps real field boundaries.
  Without it, almost no 2000 grassland or marsh pixels survived the rules (0.6 km² of marsh).
- **Fallow doesn't disqualify a wetland.** Older CDL often codes marsh as "fallow/idle" (61), so for the wetland
  class we only exclude pixels that were actually *cropped*. (Farmed NWI wetlands, `Pf`, are already excluded.)
- **"Pure pixel" rule:** a 30 m pixel on a field or wetland edge is a mix of both. We keep a pixel only if
  the whole 3×3 block around it (90 m × 90 m) belongs to the same stratum. Trade-off: this drops the tiniest
  potholes from training, so the models learn wetlands mostly from medium and large ones.

**Assumption to keep in mind:** NWI was mapped from 2017–2019 imagery. Using it for 2000 and 2010 assumes those
wetlands already existed then (and the "not cultivated in that window" rule removes ones that were being farmed).
A wetland that only formed after 2000 would be mislabelled in the 2000 training data.

In [10]:
# NWI as 0/1 rasters (Pf polygons have wet_type "Other", so they're excluded from both wetland strata)
nwi_any = ee.Image(0).byte().paint(nwi, 1)
nwi_open = ee.Image(0).byte().paint(nwi.filter(ee.Filter.inList("wet_type", ["Lake", "Freshwater Pond"])), 1)
nwi_emergent = ee.Image(0).byte().paint(nwi.filter(ee.Filter.eq("wet_type", "Freshwater Emergent Wetland")), 1)

# Final classes the models will learn
CLASS_NAMES = {0: "wetland/water", 1: "cropland", 2: "grassland/other"}
# One colour per class, used in every map and chart (blue, orange, aqua: checked to stay
# distinguishable for the common types of colour blindness)
CLASS_PALETTE = ["2a78d6", "eb6834", "1baf7a"]
# Sampling strata: two wetland sub-groups that both become class 0
STRATUM_NAMES = {0: "wetland: open water", 1: "cropland", 2: "grassland/other", 3: "wetland: emergent marsh"}
STRATUM_TO_CLASS = [0, 1, 2, 0]


def pure(mask):
    """Keep a pixel only if its whole 3x3 neighbourhood is in the mask (computed on the 30 m CDL grid)."""
    return mask.reproject(cdl_proj).focalMin(radius=1, kernelType="square", units="pixels")


def smoothed_cdl(year):
    """CDL crop codes after a 3x3 majority filter, which removes isolated 'speckle' pixels."""
    return cdl_cropland(year).reproject(cdl_proj).focalMode(radius=1, kernelType="square", units="pixels")


def label_strata(year):
    """One-band image of training strata (0-3) for a snapshot, using smoothed CDL for year-1..year+1 and NWI."""
    crops = ee.ImageCollection([smoothed_cdl(y) for y in (year - 1, year, year + 1)])
    cultivated = crops.map(is_cultivated)
    cropped = crops.map(lambda im: is_cultivated(im).And(im.neq(61)))  # cultivated, but fallow doesn't count

    always_cultivated = cultivated.min()       # 1 only if cultivated every year
    never_cultivated = cultivated.max().Not()  # 1 only if cultivated in no year
    never_cropped = cropped.max().Not()        # 1 only if never cropped (fallow/idle allowed)
    ever_fallow = crops.map(lambda im: im.eq(61)).max()
    ever_water_wetland = crops.map(
        lambda im: im.remap(WATER_WETLAND_CODES, [1] * len(WATER_WETLAND_CODES), 0)).max()

    open_water = pure(nwi_open.And(never_cropped))
    emergent = pure(nwi_emergent.And(never_cropped))
    cropland = pure(always_cultivated.And(ever_fallow.Not()).And(nwi_any.Not()))
    grass = pure(never_cultivated.And(ever_water_wetland.Not()).And(nwi_any.Not()))

    return (
        ee.Image(0).where(cropland, 1).where(grass, 2).where(emergent, 3)
        .updateMask(open_water.Or(cropland).Or(grass).Or(emergent))
        .rename("stratum").reproject(cdl_proj).clip(aoi)
    )


strata = {year: label_strata(year) for year in SNAPSHOTS}

# How much candidate area does each stratum have in each snapshot? (km², 30 m pixels = 900 m²)
candidates = {}
for year, img in strata.items():
    h = img.reduceRegion(ee.Reducer.frequencyHistogram(), aoi, crs=cdl_proj, scale=30,
                         maxPixels=1e10).get("stratum").getInfo()
    candidates[year] = {STRATUM_NAMES[int(k)]: v * 900 / 1e6 for k, v in h.items()}
print("Candidate area per stratum (km²):")
pd.DataFrame(candidates).loc[list(STRATUM_NAMES.values())].round(1)

Candidate area per stratum (km²):


,2000,2010,2020
wetland: open water,96.1,99.9,100.4
cropland,442.3,751.6,742.3
grassland/other,179.3,493.3,388.5
wetland: emergent marsh,7.9,9.5,11.1


## Step 3c — Sample training points and attach each snapshot's Landsat values

For each snapshot, `stratifiedSample` draws a set number of random points from each stratum: **150 open water +
150 emergent marsh (= 300 wetland), 300 cropland, 300 grassland/other**. Equal class sizes keep a model from being
swamped by the most common class. The fixed `seed` makes the draw repeatable. Each point then gets its final
`class` (both wetland strata → class 0) while keeping its `stratum` for checking.

Then `sampleRegions` reads **that snapshot's** composite under each point: the six reflectance bands plus NDVI,
MNDWI and TC wetness. These nine numbers are the **features** the random forest learns from. Points on a pixel with
no clear Landsat data are dropped automatically, so we print the final counts.

In [11]:
FEATURES = BANDS + ["ndvi", "mndwi", "tcw"]
STRATUM_POINTS = {0: 150, 1: 300, 2: 300, 3: 150}


def sample_training(year):
    """Stratified random points for one snapshot, with that snapshot's nine Landsat features attached."""
    points = strata[year].stratifiedSample(
        numPoints=0,  # ignored: classPoints below sets the count per stratum
        classBand="stratum", classValues=list(STRATUM_POINTS), classPoints=list(STRATUM_POINTS.values()),
        region=aoi, projection=cdl_proj, scale=30, seed=42, geometries=True,
    )
    # Map each stratum to its final class (0 open water -> 0, 3 emergent -> 0)
    points = points.map(lambda f: f.set("class", ee.List(STRATUM_TO_CLASS).get(f.get("stratum"))))
    return composites[year].select(FEATURES).sampleRegions(
        collection=points, properties=["class", "stratum"], projection=cdl_proj, scale=30, geometries=True,
    )


training, training_gdf = {}, {}
for year in SNAPSHOTS:
    training[year] = sample_training(year)
    # Bring the samples to the client as a GeoDataFrame (~900 rows) and save them for reproducibility
    gdf = geemap.ee_to_gdf(training[year])
    gdf["class_name"] = gdf["class"].map(CLASS_NAMES)
    gdf["stratum_name"] = gdf["stratum"].map(STRATUM_NAMES)
    gdf.to_file(f"outputs/training_points_{year}.geojson", driver="GeoJSON")
    training_gdf[year] = gdf

print("Training points per stratum:")
print(pd.DataFrame({y: g["stratum_name"].value_counts() for y, g in training_gdf.items()}).to_string(), "\n")

# Sanity check: average values per stratum and snapshot should look physically sensible
pd.concat({y: g.groupby("stratum_name")[["ndvi", "mndwi", "tcw", "nir", "swir1"]].mean()
           for y, g in training_gdf.items()}, names=["snapshot"]).round(3)

Training points per stratum:
                         2000  2010  2020
stratum_name                             
cropland                  300   300   300
grassland/other           300   300   300
wetland: open water       150   150   150
wetland: emergent marsh   150   150   150 



ndvi  mndwi    tcw    nir  swir1
snapshot stratum_name                                              
2000     cropland                 0.296 -0.522 -0.265  0.198  0.292
         grassland/other          0.516 -0.483 -0.178  0.299  0.271
         wetland: emergent marsh  0.339 -0.230 -0.074  0.155  0.128
         wetland: open water      0.005  0.204 -0.006  0.051  0.037
2010     cropland                 0.231 -0.526 -0.297  0.203  0.332
         grassland/other          0.477 -0.501 -0.189  0.269  0.272
         wetland: emergent marsh  0.275 -0.242 -0.077  0.133  0.125
         wetland: open water     -0.051  0.315  0.002  0.042  0.027
2020     cropland                 0.297 -0.527 -0.297  0.241  0.341
         grassland/other          0.529 -0.549 -0.206  0.289  0.285
         wetland: emergent marsh  0.404 -0.440 -0.152  0.226  0.220
         wetland: open water     -0.240  0.425  0.003  0.020  0.017

## Step 3d — Check the labels against NAIP aerial photos

Automated labels can still be wrong (e.g., a dry pothole that NWI calls wetland, or a CDL pixel in a farmyard),
so we **look** at a sample. NAIP is 0.6–1 m aerial photography. It's available here for 2020 and 2010 (but not for
2000; the earliest NAIP for this watershed is 2003), so we check those two snapshots.
Note that NAIP is flown in **summer**, while our Landsat composites are **spring**, so ponds may look smaller
and crops greener in NAIP than in Landsat.

1. **Static chip grids** (saved to `outputs/figures/training_points_naip_check_<year>.png`): 5 random points per
   stratum, each shown as a 240 m × 240 m NAIP chip with the point at the red cross.
2. **Interactive map**: all 2020 training points over NAIP 2020 (blue = wetland/water, orange = cropland,
   aqua = grassland/other). Zoom in anywhere to spot-check.

In [12]:
import io
import urllib.request

import matplotlib.pyplot as plt
from PIL import Image


def naip_mosaic(year):
    return (ee.ImageCollection("USDA/NAIP/DOQQ").filterBounds(aoi)
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01").mosaic())


def naip_chip_grid(year, n_per_stratum=5):
    """Plot NAIP chips around random training points of one snapshot and save the figure."""
    naip = naip_mosaic(year)
    check = training_gdf[year].groupby("stratum").sample(n_per_stratum, random_state=1)
    fig, axes = plt.subplots(4, n_per_stratum, figsize=(2.6 * n_per_stratum, 10.8))
    for row, stratum in enumerate([0, 3, 1, 2]):  # both wetland rows first
        group = check[check["stratum"] == stratum]
        for col, (_, pt) in enumerate(group.iterrows()):
            box = ee.Geometry.Point([pt.geometry.x, pt.geometry.y]).buffer(120).bounds()
            url = naip.getThumbURL({"bands": ["R", "G", "B"], "min": 0, "max": 255,
                                    "region": box, "dimensions": 256, "format": "png"})
            chip = Image.open(io.BytesIO(urllib.request.urlopen(url).read()))
            ax = axes[row, col]
            ax.imshow(chip)
            ax.plot(chip.width / 2, chip.height / 2, "r+", markersize=14, markeredgewidth=2)
            ax.set_xticks([]); ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(STRATUM_NAMES[stratum].replace(": ", ":\n"), fontsize=11)
    fig.suptitle(f"{year} training point check: NAIP {year}, 240 m chips, point at red cross", fontsize=12)
    fig.tight_layout()
    fig.savefig(f"outputs/figures/training_points_naip_check_{year}.png", dpi=150)
    plt.show()


for year in (2020, 2010):
    naip_chip_grid(year)

In [13]:
m3 = geemap.Map(basemap="Esri.WorldImagery")
m3.centerObject(aoi, 10)
m3.addLayer(naip_mosaic(2020), {"bands": ["R", "G", "B"], "min": 0, "max": 255}, "NAIP 2020")
m3.addLayer(nwi.style(color="00ffff", fillColor="00000000", width=1), {}, "NWI outlines", shown=False)

for cls, color in enumerate(CLASS_PALETTE):
    pts = training[2020].filter(ee.Filter.eq("class", cls))
    m3.addLayer(pts.style(color=color, pointSize=4, fillColor=color), {}, f"2020 {CLASS_NAMES[cls]} points")

m3.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m3

## Step 4a — Split each snapshot's points: 70% to train, 30% to test

We hide 30% of the points from each model while it learns, then use them to check how well it does on points
it has never seen. `randomColumn` gives every point a random number between 0 and 1; points below 0.7 go to
training and the rest to testing. The seed makes the split repeatable.

**Caveat:** this is a *random* split, so some test points sit close to training points (e.g. in the same lake).
Nearby pixels look alike, which makes the test a bit easier than classifying a brand-new area. The accuracies
below are therefore probably somewhat **optimistic**.

In [14]:
train_sets, test_sets = {}, {}
split_counts = {}
for year in SNAPSHOTS:
    split = training[year].randomColumn("random", seed=7)
    train_sets[year] = split.filter(ee.Filter.lt("random", 0.7))
    test_sets[year] = split.filter(ee.Filter.gte("random", 0.7))
    for name, fc in (("train", train_sets[year]), ("test", test_sets[year])):
        h = fc.aggregate_histogram("class").getInfo()
        split_counts[(year, name)] = {CLASS_NAMES[int(k)]: v for k, v in h.items()}

split_counts = pd.DataFrame(split_counts)
split_counts.loc["total"] = split_counts.sum()
split_counts

2000       2010       2020     
                train test train test train test
wetland/water     207   93   206   94   222   78
cropland          221   79   205   95   201   99
grassland/other   203   97   203   97   208   92
total             631  269   614  286   631  269

## Step 4b — Train one random forest per snapshot

A **random forest** is a collection of decision trees. Each tree is trained on a random subset of the points and
features and learns a series of yes/no questions (e.g. *"Is MNDWI above 0.1?"*). To classify a pixel, every
tree votes and the majority wins. It's a popular choice for land cover because it handles many features,
needs little tuning, and is hard to overfit badly.

We train **one model per snapshot**, each on its own year's points and imagery, so each model learns what the
classes looked like in that period (sensor, weather and farming practices all differ between 2000 and 2020).
Each uses 200 trees and Earth Engine's defaults for everything else. The table shows which features each model
relied on most.

In [15]:
models = {}
importance = {}
for year in SNAPSHOTS:
    models[year] = ee.Classifier.smileRandomForest(numberOfTrees=200, seed=42).train(
        features=train_sets[year], classProperty="class", inputProperties=FEATURES,
    )
    imp = pd.Series(models[year].explain().get("importance").getInfo())
    importance[year] = 100 * imp / imp.sum()

# Which features did each forest rely on most? (% of total importance; higher = more useful)
pd.DataFrame(importance).round(1).sort_values(2020, ascending=False)

,2000,2010,2020
swir2,14.2,15.2,14.2
nir,15.0,13.8,13.1
ndvi,10.4,11.1,12.7
mndwi,9.8,10.7,11.1
tcw,12.8,12.2,11.0
red,8.5,8.1,10.4
swir1,12.2,12.4,9.6
blue,8.5,8.1,9.1
green,8.5,8.5,8.7


## Step 4c — Test on the held-out 30%: confusion matrices and accuracy

The **confusion matrix** compares the true class (rows) with the model's prediction (columns). Numbers on the
diagonal are correct; everything off the diagonal is a mistake, and *where* it sits tells you which classes get mixed up.

- **Overall accuracy**: share of all test points classified correctly.
- **Producer's accuracy** (per class, read across a row): of the points that really are wetland, what share did
  the model find? Low = the class gets *missed* (omission error).
- **User's accuracy** (per class, read down a column): of the points the model *called* wetland, what share
  really are? Low = the class gets *over-predicted* (commission error).
- **Kappa**: agreement corrected for chance (1 = perfect, 0 = no better than random).

In [16]:
labels_order = [CLASS_NAMES[i] for i in range(3)]
tested = {}
summary_rows = []
for year in SNAPSHOTS:
    tested[year] = test_sets[year].classify(models[year])
    em = tested[year].errorMatrix("class", "classification")

    cm = pd.DataFrame(em.getInfo(), index=labels_order, columns=labels_order)
    cm.index.name, cm.columns.name = "true class", "predicted class"
    oa, kappa = em.accuracy().getInfo(), em.kappa().getInfo()
    producers = [row[0] for row in em.producersAccuracy().getInfo()]
    users = em.consumersAccuracy().getInfo()[0]

    cm.to_csv(f"outputs/confusion_matrix_{year}.csv")
    row = {"snapshot": year, "test_points": int(cm.values.sum()),
           "overall_accuracy": oa, "kappa": kappa}
    for i, name in enumerate(labels_order):
        row[f"producers_{name}"] = producers[i]
        row[f"users_{name}"] = users[i]
    summary_rows.append(row)

    print(f"===== {year}: overall accuracy {oa:.3f}, kappa {kappa:.3f} ({int(cm.values.sum())} test points)")
    print(cm.to_string(), "\n")

accuracy = pd.DataFrame(summary_rows).set_index("snapshot").round(3)
accuracy.to_csv("outputs/accuracy_by_snapshot.csv")
accuracy.T

===== 2000: overall accuracy 0.955, kappa 0.933 (269 test points)
predicted class  wetland/water  cropland  grassland/other
true class                                               
wetland/water               84         2                7
cropland                     1        77                1
grassland/other              1         0               96 



===== 2010: overall accuracy 0.986, kappa 0.979 (286 test points)
predicted class  wetland/water  cropland  grassland/other
true class                                               
wetland/water               92         0                2
cropland                     1        94                0
grassland/other              0         1               96 



===== 2020: overall accuracy 0.933, kappa 0.899 (269 test points)
predicted class  wetland/water  cropland  grassland/other
true class                                               
wetland/water               69         0                9
cropland                     1        92                6
grassland/other              2         0               90 



snapshot,2000,2010,2020
test_points,269.000,286.000,269.000
overall_accuracy,0.955,0.986,0.933
kappa,0.933,0.979,0.899
producers_wetland/water,0.903,0.979,0.885
users_wetland/water,0.977,0.989,0.958
producers_cropland,0.975,0.989,0.929
users_cropland,0.975,0.989,1.000
producers_grassland/other,0.990,0.990,0.978
users_grassland/other,0.923,0.980,0.857


## Step 4d — Which wetland type do the models miss?

The wetland class mixes open water and emergent marsh. Splitting each test result by **stratum** shows whether
a model handles both, or only the easy open water. Values are the share of test points classified correctly.

In [17]:
by_stratum = {}
for year in SNAPSHOTS:
    g = geemap.ee_to_gdf(tested[year])
    g["stratum_name"] = g["stratum"].map(STRATUM_NAMES)
    g["correct"] = g["classification"] == g["class"]
    by_stratum[(year, "n_test")] = g.groupby("stratum_name").size()
    by_stratum[(year, "share_correct")] = g.groupby("stratum_name")["correct"].mean().round(3)
pd.DataFrame(by_stratum)

2000                 2010                 2020  \
                        n_test share_correct n_test share_correct n_test   
stratum_name                                                               
cropland                    79         0.975     95         0.989     99   
grassland/other             97         0.990     97         0.990     92   
wetland: emergent marsh     51         0.824     44         0.955     40   
wetland: open water         42         1.000     50         1.000     38   

                                       
                        share_correct  
stratum_name                           
cropland                        0.929  
grassland/other                 0.978  
wetland: emergent marsh         0.775  
wetland: open water             1.000

## Step 5a — Classify each snapshot with its own model (as batch exports)

Each trained random forest now labels **every pixel** in its own snapshot's composite. Earth Engine applies the
models on its servers, so we never download the imagery.

**Why export to an asset?** Running a 200-tree forest on ~3 million pixels per snapshot is too heavy for Earth
Engine's *interactive* requests (they time out after a few minutes; an earlier attempt did exactly that).
Instead we submit **batch tasks** that run in the background with no time limit and save each classified map
as an Earth Engine **asset** (a raster stored in our project). Every later step reads these saved maps, which is
faster and guarantees all steps use the exact same classification.

Export settings worth knowing:
- **Grid:** the CDL's 30 m Albers equal-area grid (EPSG:5070), so every pixel is exactly 900 m² and lines up with CDL.
- **`pyramidingPolicy: mode`:** when Earth Engine shrinks the map for zoomed-out views, it takes the most common
  class instead of averaging class codes (averaging "wetland 0" and "grassland 2" would give a fake "cropland 1").
- If an asset already exists we reuse it. Set `REBUILD = True` after changing the training data or models.

In [18]:
import time

ASSET_ROOT = "projects/agriculture-wetland-loss/assets"
REBUILD = False  # set True to delete and re-run the exports (e.g. after changing the models)

grid = cdl_proj.getInfo()  # {'crs': ..., 'transform': [...]} of the 30 m CDL grid
existing = {a["id"] for a in ee.data.listAssets({"parent": ASSET_ROOT}).get("assets", [])}

tasks = []
for year, img in composites.items():
    asset_id = f"{ASSET_ROOT}/landcover_{year}"
    if asset_id in existing and not REBUILD:
        print(f"{year}: asset exists, reusing {asset_id}")
        continue
    if asset_id in existing:
        ee.data.deleteAsset(asset_id)
    task = ee.batch.Export.image.toAsset(
        image=img.select(FEATURES).classify(models[year]).rename("class").toByte(),
        description=f"landcover_{year}",
        assetId=asset_id,
        region=aoi,
        crs=grid["crs"],
        crsTransform=grid["transform"],
        maxPixels=1e10,
        pyramidingPolicy={"class": "mode"},
    )
    task.start()
    tasks.append((year, task))
    print(f"{year}: export started -> {asset_id}")

# Wait for the batch tasks to finish (you can also watch them in the Code Editor's Tasks tab)
while tasks and any(t.status()["state"] in ("UNSUBMITTED", "READY", "RUNNING") for _, t in tasks):
    time.sleep(30)
for year, t in tasks:
    s = t.status()
    print(f"{year}: {s['state']}", s.get("error_message", ""))
    assert s["state"] == "COMPLETED", f"Export for {year} did not complete"

classified = {year: ee.Image(f"{ASSET_ROOT}/landcover_{year}") for year in SNAPSHOTS}
print("Classified snapshots:", list(classified))

2000: asset exists, reusing projects/agriculture-wetland-loss/assets/landcover_2000
2010: asset exists, reusing projects/agriculture-wetland-loss/assets/landcover_2010
2020: asset exists, reusing projects/agriculture-wetland-loss/assets/landcover_2020
Classified snapshots: [2000, 2010, 2020]


## Step 5b — Sanity check: does each map agree with its reference data?

The test accuracies only covered "easy", pure pixels. Here we compare each **whole map** with the reference layers
for its own period. Every 30 m pixel gets a *reference* category from that snapshot year's CDL plus NWI:

- **NWI wetland** (non-farmed NWI polygon)
- **CDL fallow/idle** (CDL 61; excluded from training, so we don't know yet how the models treat it)
- **CDL other cultivated** (cultivated, not fallow, not NWI)
- **Everything else** (mostly grassland/pasture, developed, trees)

and we cross-tabulate the area of each reference category by the predicted class. This is **not** an accuracy
assessment (the reference layers have their own errors, and NWI is from 2017–2019), but large disagreements would
point to a problem the point test missed. The `total` rows give each map's class areas.

In [19]:
nwi_unfarmed = ee.Image(0).byte().paint(nwi.filter(ee.Filter.eq("farmed", 0)), 1)
REF_NAMES = {0: "NWI wetland", 1: "CDL fallow/idle", 2: "CDL other cultivated", 3: "everything else"}


def reference_check(year):
    """Area (km²) of each reference category by predicted class, for one snapshot."""
    crop = cdl_cropland(year)
    reference = (
        ee.Image(3)
        .where(is_cultivated(crop), 2)
        .where(crop.eq(61), 1)
        .where(nwi_unfarmed, 0)  # NWI takes priority where layers overlap
    )
    # Combine into one code (ref * 10 + predicted class) and count pixels of each combination
    combo = reference.multiply(10).add(classified[year]).rename("combo")
    hist = combo.reduceRegion(
        ee.Reducer.frequencyHistogram(), aoi, crs=cdl_proj, scale=30, maxPixels=1e10
    ).get("combo").getInfo()
    rows = [{"reference": REF_NAMES[int(float(k)) // 10],
             "predicted": CLASS_NAMES[int(float(k)) % 10],
             "km2": v * 900 / 1e6} for k, v in hist.items()]
    table = pd.DataFrame(rows).pivot_table(index="reference", columns="predicted", values="km2",
                                           aggfunc="sum", fill_value=0)
    table = table.reindex(index=list(REF_NAMES.values()), columns=list(CLASS_NAMES.values()), fill_value=0)
    table.loc["total"] = table.sum()
    return table


ref_km2 = {year: reference_check(year) for year in SNAPSHOTS}
ref_pct = {year: 100 * t.div(t.sum(axis=1), axis=0) for year, t in ref_km2.items()}

print("Area (km²) by reference category (rows) and predicted class (columns):")
print(pd.concat(ref_km2, names=["snapshot"]).round(1).to_string(), "\n")
print("Row percentages (of each reference category, what share did the model call each class?):")
print(pd.concat(ref_pct, names=["snapshot"]).round(1).to_string())

Area (km²) by reference category (rows) and predicted class (columns):
predicted                      wetland/water  cropland  grassland/other
snapshot reference                                                     
2000     NWI wetland                   308.7      13.7             29.9
         CDL fallow/idle                25.1      71.9            131.8
         CDL other cultivated           62.0    1020.3            194.5
         everything else               104.3     154.6            655.5
         total                         500.2    1260.6           1011.8
2010     NWI wetland                   318.4       8.4             25.5
         CDL fallow/idle                 0.8       1.9              0.4
         CDL other cultivated           46.6    1096.0             63.9
         everything else               176.4     200.7            833.5
         total                         542.2    1307.0            923.3
2020     NWI wetland                   309.2       8.6           

## Step 5c — Look at the three classified maps

A static figure (saved to `outputs/figures/classified_snapshots.png`) for a quick side-by-side, then an
interactive map to zoom in. Blue = wetland/water, orange = cropland, aqua = grassland/other.

The static figure is shrunk for display (most common class per display pixel), so very small potholes disappear at this size.

In [20]:
from matplotlib.patches import Patch

class_vis = {"min": 0, "max": 2, "palette": CLASS_PALETTE}

fig, axes = plt.subplots(1, 3, figsize=(15, 5.6))
for ax, (year, img) in zip(axes, classified.items()):
    url = img.visualize(**class_vis).getThumbURL({"region": aoi, "dimensions": 800, "format": "png"})
    ax.imshow(Image.open(io.BytesIO(urllib.request.urlopen(url).read())))
    y0, y1, sensors = SNAPSHOTS[year]
    ax.set_title(f"{year}  (spring {y0}–{y1}, {'+'.join(sensors)})", fontsize=12)
    ax.axis("off")

legend = [Patch(color=f"#{c}", label=CLASS_NAMES[i]) for i, c in enumerate(CLASS_PALETTE)]
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=11, frameon=False)
fig.suptitle("Pipestem watershed (HUC8 10160002): random forest classification, one model per snapshot", fontsize=14)
fig.tight_layout(rect=[0, 0.06, 1, 0.95])
fig.savefig("outputs/figures/classified_snapshots.png", dpi=150)
plt.show()

In [21]:
m5 = geemap.Map(basemap="Esri.WorldImagery")
m5.centerObject(aoi, 10)
for year, img in classified.items():
    m5.addLayer(img, class_vis, f"{year} classified", shown=(year in (2000, 2020)))
m5.addLayer(nwi.style(color="00ffff", fillColor="00000000", width=1), {}, "NWI outlines", shown=False)
m5.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": "yellow"}, "Pipestem boundary")
m5.add_legend(title="Class", legend_dict={CLASS_NAMES[i]: c for i, c in enumerate(CLASS_PALETTE)})
m5

## Step 6a — Area of each class in each snapshot

We add up the area of every pixel in each class. `ee.Image.pixelArea()` gives each pixel's true area in m²; on our
equal-area 30 m grid that's 900 m² per pixel (pixels cut by the watershed boundary count only the part inside).
Grouping the sum by the class band gives one total per class.

The table shows km², the share of the watershed, and the change from 2000 to 2020. These are **mapped** areas: they
inherit the classification errors from Steps 4–5 (especially marsh vs. grassland), so treat small changes with
caution.

In [22]:
area_rows = []
for year, img in classified.items():
    groups = (
        ee.Image.pixelArea().divide(1e6).rename("km2").addBands(img)
        .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
                      geometry=aoi, crs=cdl_proj, scale=30, maxPixels=1e10)
        .get("groups").getInfo()
    )
    for g in groups:
        area_rows.append({"snapshot": year, "class": CLASS_NAMES[int(g["class"])], "km2": g["sum"]})

areas = pd.DataFrame(area_rows).pivot(index="class", columns="snapshot", values="km2")
areas = areas.loc[list(CLASS_NAMES.values())]
total_km2 = areas.sum()  # same total every year (the watershed)

area_table = areas.round(1)
for year in SNAPSHOTS:
    area_table[f"{year}_pct"] = (100 * areas[year] / total_km2[year]).round(1)
area_table["change_2000_2020_km2"] = (areas[2020] - areas[2000]).round(1)
area_table["change_2000_2020_pct"] = (100 * (areas[2020] - areas[2000]) / areas[2000]).round(1)
area_table.columns = area_table.columns.map(str)
area_table.to_csv("outputs/class_area_by_snapshot.csv")

print("Total classified area per snapshot (km²):", total_km2.round(1).to_dict())
area_table

Total classified area per snapshot (km²): {2000: 2772.5, 2010: 2772.5, 2020: 2772.5}


snapshot,2000,2010,2020,2000_pct,2010_pct,2020_pct,change_2000_2020_km2,change_2000_2020_pct
class,,,,,,,,
wetland/water,500.2,542.2,559.5,18.0,19.6,20.2,59.4,11.9
cropland,1260.6,1307.0,1414.7,45.5,47.1,51.0,154.1,12.2
grassland/other,1011.8,923.3,798.3,36.5,33.3,28.8,-213.5,-21.1


## Step 6b — Bar chart of class area by snapshot

Grouped bars: one group per snapshot, one bar per class, in the same colours as the maps. Each bar is labelled with
its area so the chart can be read without relying on colour. Saved to `outputs/figures/class_area_by_snapshot.png`.

In [23]:
import numpy as np

INK, INK_MUTED, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"

years = list(SNAPSHOTS)
x = np.arange(len(years))
bar_w = 0.26

fig, ax = plt.subplots(figsize=(9, 5.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for i, (cls, name) in enumerate(CLASS_NAMES.items()):
    values = [areas.loc[name, y] for y in years]
    bars = ax.bar(x + (i - 1) * bar_w, values, width=bar_w,
                  color=f"#{CLASS_PALETTE[cls]}", edgecolor=SURFACE, linewidth=2,  # 2px gap between bars
                  label=name, zorder=3)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + 15, f"{v:,.0f}", ha="center", va="bottom",
                fontsize=9, color=INK_MUTED)

ax.set_xticks(x, [str(y) for y in years], fontsize=11, color=INK)
ax.set_ylabel("Area (km²)", color=INK_MUTED)
ax.yaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
ax.tick_params(axis="y", colors=INK_MUTED, length=0)
ax.tick_params(axis="x", length=0)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(INK_MUTED)
ax.set_ylim(0, areas.values.max() * 1.12)
ax.legend(frameon=False, ncol=3, loc="upper left", bbox_to_anchor=(0, 1.02), fontsize=10, labelcolor=INK)
ax.set_title("Mapped land cover, Pipestem watershed (HUC8 10160002)", loc="left", fontsize=13, color=INK, pad=28)
fig.text(0.01, 0.01, "Spring Landsat composites (3-year windows), random forest per snapshot. "
         "Mapped areas, not error-adjusted.", fontsize=8.5, color=INK_MUTED)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig("outputs/figures/class_area_by_snapshot.png", dpi=150, facecolor=SURFACE)
plt.show()

## Step 7a — Persistent conversion: wetland in 2000, cropland in 2010 **and** 2020

A pixel counts as **converted** if it was classified wetland/water in 2000 and cropland in both 2010 and 2020.

Why require *both* later snapshots? Any single map has errors, and a wet field or a dry pothole can be misread in
one snapshot. Requiring cropland twice, ten years apart, filters out one-off misclassifications and short-lived
changes, so what remains is conversion that **persisted**. The trade-off: conversions that happened after 2010
are not counted.

For context we also compute:
- the **fate of all 2000 wetland**: what each 2000 wetland pixel was classified as in 2020
- the **reverse transition** (cropland in 2000, wetland in 2010 and 2020), since wetlands also expand in wet years

In [24]:
c2000, c2010, c2020 = (classified[y] for y in (2000, 2010, 2020))
converted = c2000.eq(0).And(c2010.eq(1)).And(c2020.eq(1)).rename("converted")
reverse = c2000.eq(1).And(c2010.eq(0)).And(c2020.eq(0)).rename("reverse")


def area_km2(mask):
    """Total area (km²) of pixels where mask is 1."""
    return ee.Image.pixelArea().divide(1e6).updateMask(mask).reduceRegion(
        ee.Reducer.sum(), aoi, crs=cdl_proj, scale=30, maxPixels=1e10).get("area").getInfo()


wetland_2000_km2 = area_km2(c2000.eq(0))
converted_km2 = area_km2(converted)
reverse_km2 = area_km2(reverse)

print(f"Wetland/water in 2000:                          {wetland_2000_km2:8.1f} km²")
print(f"Converted (wetland 2000 -> cropland 2010 & 2020): {converted_km2:8.1f} km²  "
      f"= {100 * converted_km2 / wetland_2000_km2:.1f}% of 2000 wetland")
print(f"Reverse (cropland 2000 -> wetland 2010 & 2020):   {reverse_km2:8.1f} km²")

# Fate of 2000 wetland: what was it classified as in 2020?
fate = {CLASS_NAMES[k]: area_km2(c2000.eq(0).And(c2020.eq(k))) for k in CLASS_NAMES}
fate_table = pd.DataFrame({"km2": fate})
fate_table["pct_of_2000_wetland"] = 100 * fate_table["km2"] / wetland_2000_km2
fate_table.index.name = "2020 class of 2000 wetland"
fate_table.round(1)

Wetland/water in 2000:                             500.2 km²
Converted (wetland 2000 -> cropland 2010 & 2020):     19.4 km²  = 3.9% of 2000 wetland
Reverse (cropland 2000 -> wetland 2010 & 2020):       26.2 km²


,km2,pct_of_2000_wetland
2020 class of 2000 wetland,,
wetland/water,393.3,78.6
cropland,44.0,8.8
grassland/other,62.9,12.6


## Step 7b — How believable are the converted pixels?

Three quick checks on the converted pixels, using data the classification never saw directly:

1. **CDL agreement:** does the (raw, unsmoothed) Cropland Data Layer also say these pixels were cultivated in 2010
   and in 2020?
2. **NWI overlap:** NWI was mapped from 2017–2019 imagery. Converted pixels inside a **farmed wetland** (`Pf`)
   polygon fit the story (a wetland that is now cropped). Converted pixels inside a **non-farmed** NWI wetland
   contradict it (NWI saw a wetland there recently), so those are likely classification errors.
3. **Patch size:** real drainage or plowing usually affects groups of pixels; isolated single pixels are more likely
   to be noise at field and wetland edges. We measure how much converted area sits in patches of different sizes
   (pixels touching, including diagonally).

In [25]:
conv = converted.selfMask()

# 1. CDL agreement (raw CDL, our code-based "cultivated" rule)
cdl_cult_both = is_cultivated(cdl_cropland(2010)).And(is_cultivated(cdl_cropland(2020)))
cdl_check = {
    "CDL cultivated in 2010 and 2020": area_km2(converted.And(cdl_cult_both)),
    "CDL not cultivated in at least one of them": area_km2(converted.And(cdl_cult_both.Not())),
}

# 2. NWI overlap
nwi_farmed = ee.Image(0).byte().paint(nwi.filter(ee.Filter.eq("farmed", 1)), 1)
nwi_check = {
    "inside NWI farmed wetland (Pf)": area_km2(converted.And(nwi_farmed)),
    "inside NWI non-farmed wetland": area_km2(converted.And(nwi_unfarmed)),
    "outside NWI": area_km2(converted.And(nwi_any.Not())),
}

# 3. Patch size (connected converted pixels, 8-neighbour), capped at 100 pixels for speed
patch_px = conv.reproject(cdl_proj).connectedPixelCount(maxSize=100, eightConnected=True)
patch_check = {
    "single pixel (0.09 ha)": area_km2(patch_px.eq(1)),
    "2-10 pixels": area_km2(patch_px.gte(2).And(patch_px.lte(10))),
    "11+ pixels (≥ 1 ha)": area_km2(patch_px.gte(11)),
}

checks = pd.concat({
    "1. CDL agreement": pd.Series(cdl_check),
    "2. NWI overlap": pd.Series(nwi_check),
    "3. patch size": pd.Series(patch_check),
}).rename("km2").to_frame()
checks["pct_of_converted"] = 100 * checks["km2"] / converted_km2
checks.round(1)

km2  \
1. CDL agreement CDL cultivated in 2010 and 2020             15.3   
                 CDL not cultivated in at least one of them   4.1   
2. NWI overlap   inside NWI farmed wetland (Pf)               0.4   
                 inside NWI non-farmed wetland                0.4   
                 outside NWI                                 18.7   
3. patch size    single pixel (0.09 ha)                       3.3   
                 2-10 pixels                                  7.0   
                 11+ pixels (≥ 1 ha)                          9.2   

                                                             pct_of_converted  
1. CDL agreement CDL cultivated in 2010 and 2020                         78.9  
                 CDL not cultivated in at least one of them              21.1  
2. NWI overlap   inside NWI farmed wetland (Pf)                           1.9  
                 inside NWI non-farmed wetland                            2.0  
                 outside NWI                                             96.0  
3. patch size    single pixel (0.09 ha)                                  17.1  
                 2-10 pixels                                             35.8  
                 11+ pixels (≥ 1 ha)                                     47.1

## Step 7c — Conversion map

Two figures, saved to `outputs/figures/`:

- **`conversion_map.png`** — the whole watershed: 2000 wetland in light blue, converted pixels in red.
  Converted pixels are tiny (30 m) at this scale, so for display only, each screen cell is drawn red if *any*
  converted pixel falls inside it. This makes them visible but **exaggerates their extent**; the areas above are
  the real numbers.
- **`conversion_examples.png`** — the two 2 km × 2 km squares with the most converted area, shown on NAIP aerial
  photos from the earliest available year (before) and 2020 (after), with converted pixels outlined. This lets you
  judge by eye whether the "conversions" look real.

In [26]:
CONVERTED_COLOR = "e34948"  # red: reserved for the conversion layer, not used by any class

# --- Overview map (display-only enlargement: any converted 30 m pixel within a 150 m cell shows red) ---
display_conv = (conv.reproject(cdl_proj)
                .reduceResolution(ee.Reducer.max(), maxPixels=64)
                .reproject(cdl_proj.atScale(150)))
overview = ee.ImageCollection([
    ee.Image(1).clip(aoi).visualize(palette=["e4e3df"]),                                # watershed (gray)
    c2000.eq(0).selfMask().visualize(palette=["a9c8ef"]),                              # 2000 wetland (light blue)
    display_conv.visualize(palette=[CONVERTED_COLOR]),                                 # converted (red)
    ee.Image().paint(aoi, 0, 1).visualize(palette=["52514e"]),                         # boundary
]).mosaic()
url = overview.getThumbURL({"region": aoi.bounds(), "dimensions": 1000, "format": "png"})

fig, ax = plt.subplots(figsize=(9, 7.5), facecolor=SURFACE)
ax.imshow(Image.open(io.BytesIO(urllib.request.urlopen(url).read())))
ax.axis("off")
legend = [Patch(color="#a9c8ef", label="wetland/water in 2000"),
          Patch(color=f"#{CONVERTED_COLOR}", label="converted: wetland 2000 → cropland 2010 & 2020"),
          Patch(color="#e4e3df", label="other land")]
ax.legend(handles=legend, loc="lower left", frameon=False, fontsize=10, labelcolor=INK)
ax.set_title(f"Persistent wetland-to-cropland conversion, Pipestem watershed, 2000–2020\n"
             f"{converted_km2:.1f} km² converted ({100 * converted_km2 / wetland_2000_km2:.1f}% of 2000 wetland)",
             loc="left", fontsize=12, color=INK)
fig.text(0.01, 0.01, "Converted pixels enlarged for display (any converted 30 m pixel within a 150 m cell is shown). "
         "Areas are computed at 30 m.", fontsize=8.5, color=INK_MUTED)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig("outputs/figures/conversion_map.png", dpi=150, facecolor=SURFACE)
plt.show()

# --- Hotspots: the two 2 km cells with the most converted area ---
cells = aoi.coveringGrid(cdl_proj, 2000)
cells = ee.Image.pixelArea().divide(1e6).updateMask(converted).reduceRegions(
    collection=cells, reducer=ee.Reducer.sum().setOutputs(["conv_km2"]), crs=cdl_proj, scale=30)
top_cells = cells.sort("conv_km2", False).limit(2).getInfo()["features"]


def naip_year_for(region, years):
    """First year in `years` with NAIP imagery covering the region."""
    for y in years:
        if ee.ImageCollection("USDA/NAIP/DOQQ").filterBounds(region).filterDate(f"{y}-01-01", f"{y + 1}-01-01").size().getInfo():
            return y
    return None


outline = conv.reproject(cdl_proj).unmask(0).focalMax(1, "square", "pixels").subtract(conv.unmask(0)).selfMask()
fig, axes = plt.subplots(len(top_cells), 2, figsize=(10, 5 * len(top_cells)), facecolor=SURFACE)
for row, cell in enumerate(top_cells):
    region = ee.Geometry(cell["geometry"])
    before = naip_year_for(region, range(2003, 2010))
    for col, year in enumerate((before, 2020)):
        img = ee.ImageCollection([
            naip_mosaic(year).visualize(bands=["R", "G", "B"], min=0, max=255),
            outline.visualize(palette=[CONVERTED_COLOR]),
        ]).mosaic()
        url = img.getThumbURL({"region": region, "dimensions": 500, "format": "png"})
        ax = axes[row, col]
        ax.imshow(Image.open(io.BytesIO(urllib.request.urlopen(url).read())))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"Hotspot {row + 1} — NAIP {year}", fontsize=11, color=INK)
    axes[row, 0].set_ylabel(f"{cell['properties']['conv_km2']:.2f} km² converted\nin this 2×2 km cell",
                            fontsize=10, color=INK_MUTED)
fig.suptitle("Conversion hotspots: converted pixels outlined in red (before vs. after)", fontsize=12, color=INK)
fig.tight_layout()
fig.savefig("outputs/figures/conversion_examples.png", dpi=150, facecolor=SURFACE)
plt.show()

## Step 8a — Yearly precipitation for the watershed (PRISM)

Prairie potholes fill and dry with the weather, so wetland area in any snapshot depends partly on how wet those
years were. **PRISM** (Oregon State University) provides monthly precipitation grids for the U.S. at ~4 km.
We use `OREGONSTATE/PRISM/ANm` (the plan named `OREGONSTATE/PRISM/AN81m`, which Earth Engine has deprecated and
replaced with this collection).

For each calendar year we add up the 12 monthly grids, then average over the watershed (one number per year,
in mm). We also total **April–June** separately, since that's the season our Landsat composites cover.
Range: 1985–2025 (complete years only), which shows the snapshot windows in a longer context.

In [27]:
prism = ee.ImageCollection("OREGONSTATE/PRISM/ANm").select("ppt")
PRISM_YEARS = list(range(1985, 2026))
PRISM_SCALE = 4638.3  # PRISM's native ~4 km pixel size, in metres


def watershed_ppt(year):
    """Watershed-mean precipitation (mm) for a calendar year and for April-June of that year."""
    months = prism.filter(ee.Filter.calendarRange(year, year, "year"))
    annual = months.sum().rename("annual_mm")
    spring = months.filter(ee.Filter.calendarRange(4, 6, "month")).sum().rename("apr_jun_mm")
    stats = annual.addBands(spring).reduceRegion(ee.Reducer.mean(), aoi, scale=PRISM_SCALE)
    return ee.Feature(None, stats).set({"year": year, "n_months": months.size()})


ppt_fc = ee.FeatureCollection([watershed_ppt(y) for y in PRISM_YEARS])
precip = pd.DataFrame(ppt_fc.getInfo()["features"]).properties.apply(pd.Series).set_index("year")
assert (precip["n_months"] == 12).all(), "a year is missing PRISM months"
precip.index = precip.index.astype(int)
precip = precip[["annual_mm", "apr_jun_mm"]].round(1)
precip.to_csv("outputs/precipitation_annual.csv")

# Compare each snapshot window with the 1985-2025 average
long_term = precip.mean()
windows = pd.DataFrame({
    year: precip.loc[y0:y1].mean() for year, (y0, y1, _) in SNAPSHOTS.items()
}).T
windows.index.name = "snapshot window"
windows["annual_vs_avg_pct"] = 100 * (windows["annual_mm"] / long_term["annual_mm"] - 1)
windows["apr_jun_vs_avg_pct"] = 100 * (windows["apr_jun_mm"] / long_term["apr_jun_mm"] - 1)
print(f"1985–2025 average: {long_term['annual_mm']:.1f} mm/yr, April–June {long_term['apr_jun_mm']:.1f} mm")
windows.round(1)

1985–2025 average: 505.0 mm/yr, April–June 193.7 mm


,annual_mm,apr_jun_mm,annual_vs_avg_pct,apr_jun_vs_avg_pct
snapshot window,,,,
2000,562.0,220.4,11.3,13.8
2010,596.0,186.7,18.0,-3.6
2020,524.7,141.4,3.9,-27.0


## Step 8b — Precipitation chart with the snapshot windows highlighted

Bars are yearly totals; the dashed line is the 1985–2025 average; the shaded bands are the three 3-year windows
our composites come from. Saved to `outputs/figures/precipitation_chart.png`.

In [28]:
BAR_COLOR, WINDOW_SHADE = "#2a78d6", "#ecebe7"

fig, ax = plt.subplots(figsize=(11, 5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for year, (y0, y1, _) in SNAPSHOTS.items():
    ax.axvspan(y0 - 0.5, y1 + 0.5, color=WINDOW_SHADE, zorder=0)
    ax.text((y0 + y1) / 2, precip["annual_mm"].max() * 1.06, f"{year} window\n({y0}–{y1})",
            ha="center", va="bottom", fontsize=9, color=INK_MUTED)

ax.bar(precip.index, precip["annual_mm"], width=0.8, color=BAR_COLOR, edgecolor=SURFACE, linewidth=1,
       zorder=3, label="annual precipitation")
ax.axhline(long_term["annual_mm"], color=INK_MUTED, linestyle="--", linewidth=1.2, zorder=4)
ax.text(PRISM_YEARS[-1] + 0.6, long_term["annual_mm"], f"1985–2025\naverage\n{long_term['annual_mm']:.0f} mm",
        va="center", fontsize=9, color=INK_MUTED)

ax.set_xlim(PRISM_YEARS[0] - 0.7, PRISM_YEARS[-1] + 0.7)
ax.set_ylim(0, precip["annual_mm"].max() * 1.22)
ax.set_xticks(range(1985, 2026, 5))
ax.set_ylabel("Precipitation (mm per year)", color=INK_MUTED)
ax.yaxis.grid(True, color=GRID, linewidth=0.8, zorder=1)
ax.tick_params(colors=INK_MUTED, length=0)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(INK_MUTED)
ax.set_title("Annual precipitation, Pipestem watershed (HUC8 10160002), 1985–2025", loc="left",
             fontsize=13, color=INK, pad=12)
fig.text(0.01, 0.01, "PRISM monthly precipitation (OREGONSTATE/PRISM/ANm), summed per calendar year and "
         "averaged over the watershed. Shaded: Landsat snapshot windows.", fontsize=8.5, color=INK_MUTED)
fig.tight_layout(rect=[0, 0.04, 0.95, 1])
fig.savefig("outputs/figures/precipitation_chart.png", dpi=150, facecolor=SURFACE)
plt.show()